# Attention-Guided Swin Transformer for Image Compression
## MSc Dissertation 

This notebook covers the training, validation, and Rate-Distortion (R-D) curve plotting for the custom Attention-Guided Swin Transformer over the Honeybee UVG 512x512 dataset.

In [ ]:
!pip install compressai timm pytorch-msssim matplotlib lpips pandas

In [ ]:
import os
import glob
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from compressai.losses import RateDistortionLoss
from pytorch_msssim import ms_ssim
import lpips
from model import AttentionGuidedSwinCompression

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Kaggle Dataset Path
DATASET_PATH = '/kaggle/input/datasets/jeevajoji/uvg-honeybee-512x512/honeybee_512_crop/'

### Custom PyTorch Dataset

In [ ]:
class UVGDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '*.png')))
        self.transform = transform
        if len(self.image_paths) == 0:
            print(f"Warning: No PNG images found in {root_dir}")
        else:
            print(f"Found {len(self.image_paths)} images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.image_paths[idx]

transform = transforms.Compose([
    transforms.RandomCrop(256),
    transforms.ToTensor()
])

train_dataset = UVGDataset(DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Setup LPIPS (VGG) for perceptual metric
loss_fn_vgg = lpips.LPIPS(net='vgg').to(DEVICE)

### Utility Functions: Metrics and Visualizations

In [ ]:
def compute_metrics(x, x_hat, bpp_loss):
    # PSNR
    mse = torch.mean((x - x_hat) ** 2).item()
    psnr = 10 * math.log10(1.0 / mse) if mse > 0 else 100
    
    # MS-SSIM
    msssim_val = ms_ssim(x_hat, x, data_range=1.0, size_average=True).item()
    
    # LPIPS (Requires inputs in [-1, 1] format)
    x_lpips = (x * 2) - 1
    x_hat_lpips = (x_hat * 2) - 1
    lpips_val = loss_fn_vgg(x_lpips, x_hat_lpips).mean().item()
    
    return bpp_loss.item(), psnr, msssim_val, lpips_val

def save_visual_comparison(original, reconstructed, epoch, lmbda, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    # Take first image in batch
    orig_img = original[0].cpu().detach()
    recon_img = reconstructed[0].cpu().detach().clamp(0, 1)
    
    grid = make_grid([orig_img, recon_img], nrow=2)
    grid_np = grid.permute(1, 2, 0).numpy()
    
    plt.figure(figsize=(10, 5))
    plt.imshow(grid_np)
    plt.title(f"Epoch {epoch} (Lambda={lmbda}) | Left: Original, Right: Reconstructed")
    plt.axis('off')
    out_path = f"{out_dir}/epoch_{epoch}_lambda_{lmbda}.png"
    plt.savefig(out_path)
    plt.close()
    return out_path

### Training Loop for Multiple Lambdas

In [ ]:
def train_model_for_lambda(lmbda, epochs=EPOCHS):
    print(f"\n--- Training model for Lambda = {lmbda} ---")
    model = AttentionGuidedSwinCompression(N=128, M=192).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = RateDistortionLoss(lmbda=lmbda)
    
    metrics_log = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, epoch_bpp, epoch_psnr, epoch_msssim, epoch_lpips = 0.0, 0.0, 0.0, 0.0, 0.0
        
        for i, (images, _) in enumerate(train_loader):
            images = images.to(DEVICE)
            optimizer.zero_grad()
            
            out_net = model(images)
            out_criterion = criterion(out_net, images)
            
            out_criterion["loss"].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            bpp, psnr, msssim, lpips_val = compute_metrics(images, out_net['x_hat'], out_criterion["bpp_loss"])
            
            epoch_loss += out_criterion["loss"].item()
            epoch_bpp += bpp
            epoch_psnr += psnr
            epoch_msssim += msssim
            epoch_lpips += lpips_val
            
        avg_loss = epoch_loss / len(train_loader)
        avg_bpp = epoch_bpp / len(train_loader)
        avg_psnr = epoch_psnr / len(train_loader)
        avg_msssim = epoch_msssim / len(train_loader)
        avg_lpips = epoch_lpips / len(train_loader)
        
        print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | BPP: {avg_bpp:.4f} | PSNR: {avg_psnr:.2f} | MS-SSIM: {avg_msssim:.4f} | LPIPS: {avg_lpips:.4f}")
        
        # Save visual comparison side-by-side and get the saved image path
        saved_image_path = save_visual_comparison(images, out_net['x_hat'], epoch, lmbda)
        
        # Log for documentation (includes the path to the image!)
        metrics_log.append({
            "Lambda": lmbda, 
            "Epoch": epoch, 
            "Total_Loss": avg_loss, 
            "BPP": avg_bpp, 
            "PSNR": avg_psnr, 
            "MS_SSIM": avg_msssim, 
            "LPIPS": avg_lpips,
            "Image_File": saved_image_path  # <--- Added image reference to the CSV log
        })
        
    torch.save(model.state_dict(), f"swin_compress_lambda_{lmbda}.pth")
    return metrics_log


In [ ]:
lambdas = [0.0018, 0.0035, 0.0067, 0.0130, 0.0250]
all_metrics = []

# --- UNCOMMENT TO RUN ---
# for l in lambdas:
#     logs = train_model_for_lambda(l, epochs=EPOCHS)
#     all_metrics.extend(logs)
#     
# # Save all metrics to a CSV document
# df = pd.DataFrame(all_metrics)
# df.to_csv("compression_metrics_log.csv", index=False)
# print("Training complete. Metrics saved to compression_metrics_log.csv")

# # Create a Markdown report embedding the images directly for easy viewing!
# markdown_content = "# Training Log with Visual Comparisons\n\n"
# for index, row in df.iterrows():
#     markdown_content += f"## Epoch {row['Epoch']} (Lambda {row['Lambda']})\n"
#     markdown_content += f"- **BPP:** {row['BPP']:.4f}\n"
#     markdown_content += f"- **PSNR:** {row['PSNR']:.2f}\n"
#     markdown_content += f"- **MS-SSIM:** {row['MS_SSIM']:.4f}\n"
#     markdown_content += f"- **LPIPS:** {row['LPIPS']:.4f}\n\n"
#     markdown_content += f"![Comparison]({row['Image_File']})\n\n"
# 
# with open("training_report.md", "w") as f:
#     f.write(markdown_content)
# print("Markdown visual report saved to training_report.md")
